# hybrid sweep

- grid search for heuristic / cf balance
- compares accuracy and concentration


In [1]:
from __future__ import annotations

from ast import literal_eval
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "synthetic"


## Load


In [2]:
collab_table = pd.read_csv(DATA_DIR / "collaboration_training_table.csv")
collabs = pd.read_csv(DATA_DIR / "collaborations.csv")

collab_table["lastEventAt"] = pd.to_datetime(collab_table["lastEventAt"], utc=True)
collab_table["tagsKey"] = collab_table["tagsKey"].apply(literal_eval)
collabs["tagsKey"] = collabs["tagsKey"].apply(literal_eval)

feature_cols = [
    "collaboration_favorite",
    "collaboration_like",
    "submission_favorite",
    "submission_like",
    "submission_vote",
    "totalEventWeight",
]

active_collabs = collabs[collabs["status"].isin(["submission", "voting"])].copy()
active_collabs = active_collabs[["id", "projectId", "name", "status", "tagsKey", "publishedAt"]].rename(columns={"id": "collaborationId"})


## Holdout split


In [3]:
eligible_history = collab_table[collab_table["status"].isin(["submission", "voting"])].copy()
eligible_history = eligible_history.sort_values(["userId", "lastEventAt"])

holdout = eligible_history.groupby("userId").tail(1).copy()
train_history = eligible_history.merge(
    holdout[["userId", "collaborationId"]].assign(_holdout=1),
    on=["userId", "collaborationId"],
    how="left",
)
train_history = train_history[train_history["_holdout"].isna()].drop(columns="_holdout")

valid_users = train_history["userId"].value_counts()
valid_users = set(valid_users[valid_users > 0].index) & set(holdout["userId"])
train_history = train_history[train_history["userId"].isin(valid_users)].copy()
holdout = holdout[holdout["userId"].isin(valid_users)].copy()

len(valid_users), train_history.shape, holdout.shape

(239, (1434, 14), (239, 14))

## Shared components


In [4]:
train_popularity = (
    train_history
    .groupby(["collaborationId", "projectId"], as_index=False)[feature_cols]
    .sum()
)
train_popularity["popularityScore"] = (
    1.75 * train_popularity["collaboration_favorite"]
    + 0.75 * train_popularity["collaboration_like"]
    + 2.0 * train_popularity["submission_favorite"]
    + 1.0 * train_popularity["submission_like"]
    + 3.0 * train_popularity["submission_vote"]
)
train_popularity["popularityScoreNorm"] = train_popularity["popularityScore"] / (train_popularity["popularityScore"].max() or 1)
train_popularity = train_popularity[["collaborationId", "projectId", "popularityScoreNorm"]]

tag_rows = []
for row in train_history[["userId", "tagsKey", "totalEventWeight"]].itertuples(index=False):
    tags = row.tagsKey or []
    if not tags:
        continue
    per_tag_weight = row.totalEventWeight / len(tags)
    for tag in tags:
        tag_rows.append({"userId": row.userId, "tag": tag, "weight": per_tag_weight})

train_user_tag_affinity = pd.DataFrame(tag_rows).groupby(["userId", "tag"], as_index=False)["weight"].sum()
train_tag_pref = train_user_tag_affinity.groupby("userId").apply(lambda x: dict(zip(x["tag"], x["weight"]))).to_dict()

train_user_project_affinity = (
    train_history
    .groupby(["userId", "projectId"], as_index=False)["totalEventWeight"]
    .sum()
    .rename(columns={"totalEventWeight": "projectAffinity"})
)
max_project_affinity = train_user_project_affinity.groupby("userId")["projectAffinity"].transform("max")
train_user_project_affinity["projectAffinityNorm"] = train_user_project_affinity["projectAffinity"] / max_project_affinity

interaction_df = (
    train_history.groupby(["userId", "collaborationId"], as_index=False)["totalEventWeight"]
    .sum()
)
interaction_matrix = interaction_df.pivot(index="userId", columns="collaborationId", values="totalEventWeight").fillna(0.0)

X = interaction_matrix.to_numpy(dtype=float)
item_matrix = X.T
norms = np.linalg.norm(item_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1.0
item_matrix_norm = item_matrix / norms
similarity = item_matrix_norm @ item_matrix_norm.T
np.fill_diagonal(similarity, 0.0)

item_ids = interaction_matrix.columns.tolist()
sim_df = pd.DataFrame(similarity, index=item_ids, columns=item_ids)


## Candidate table


In [5]:
seen_train = train_history[["userId", "collaborationId"]].drop_duplicates().assign(seen=1)
eval_users = pd.DataFrame({"userId": sorted(valid_users)})

candidates = eval_users.assign(_k=1).merge(active_collabs.assign(_k=1), on="_k").drop(columns="_k")
candidates = candidates.merge(seen_train, on=["userId", "collaborationId"], how="left")
candidates = candidates[candidates["seen"].isna()].drop(columns="seen")
candidates = candidates.merge(train_popularity, on=["collaborationId", "projectId"], how="left")
candidates["popularityScoreNorm"] = candidates["popularityScoreNorm"].fillna(0)
candidates = candidates.merge(
    train_user_project_affinity[["userId", "projectId", "projectAffinityNorm"]],
    on=["userId", "projectId"],
    how="left",
)
candidates["projectAffinityNorm"] = candidates["projectAffinityNorm"].fillna(0)

def tag_score(user_id, tags):
    pref = train_tag_pref.get(user_id, {})
    if not tags:
        return 0.0
    return sum(pref.get(tag, 0.0) for tag in tags) / len(tags)

candidates["tagScoreRaw"] = [tag_score(u, tags) for u, tags in zip(candidates["userId"], candidates["tagsKey"])]
max_tag = candidates.groupby("userId")["tagScoreRaw"].transform("max").replace(0, 1)
candidates["tagScoreNorm"] = candidates["tagScoreRaw"] / max_tag

pop_map = dict(zip(train_popularity["collaborationId"], train_popularity["popularityScoreNorm"]))
cf_rows = []
active_ids = candidates["collaborationId"].unique().tolist()
for user_id, row in interaction_matrix.iterrows():
    seen_weights = row[row > 0]
    seen_ids = set(seen_weights.index)
    user_scores = {}
    if len(seen_weights) > 0:
        seen_vector = seen_weights.to_numpy(dtype=float)
        sims = sim_df[seen_weights.index].mul(seen_vector, axis=1).sum(axis=1)
        user_scores = sims.to_dict()
    for collab_id in active_ids:
        if collab_id in seen_ids:
            continue
        cf_rows.append({
            "userId": user_id,
            "collaborationId": collab_id,
            "cfScoreRaw": float(user_scores.get(collab_id, 0.0)),
            "cfPopularityNorm": float(pop_map.get(collab_id, 0.0)),
        })

cf_scores = pd.DataFrame(cf_rows)
max_cf = cf_scores.groupby("userId")["cfScoreRaw"].transform("max").replace(0, 1)
cf_scores["cfScoreNorm"] = cf_scores["cfScoreRaw"] / max_cf

candidates = candidates.merge(cf_scores, on=["userId", "collaborationId"], how="left")
candidates[["cfScoreRaw", "cfPopularityNorm", "cfScoreNorm"]] = candidates[["cfScoreRaw", "cfPopularityNorm", "cfScoreNorm"]].fillna(0)

history_tags = train_history.groupby("userId")["tagsKey"].apply(lambda rows: set(t for arr in rows for t in arr)).to_dict()
history_projects = train_history.groupby("userId")["projectId"].apply(lambda s: set(s)).to_dict()
holdout_eval = holdout[["userId", "collaborationId"]].rename(columns={"collaborationId": "heldOutCollaborationId"})

candidates.head()

,userId,collaborationId,projectId,name,status,tagsKey,publishedAt,popularityScoreNorm,projectAffinityNorm,tagScoreRaw,tagScoreNorm,cfScoreRaw,cfPopularityNorm,cfScoreNorm
0,user-0001,collab-001-01,project-001,Collaboration 1-1,voting,"[acid, breakbeat, cinematic]",2026-04-28T14:26:30Z,0.716581,0.0,2.305556,0.461111,1.462580,0.716581,0.390676
1,user-0001,collab-001-02,project-001,Collaboration 1-2,voting,"[breakbeat, lofi, organic]",2025-12-17T02:06:25Z,0.894814,0.0,0.638889,0.127778,0.994962,0.894814,0.265769
2,user-0001,collab-002-01,project-002,Collaboration 2-1,submission,"[ambient, drum-and-bass]",2026-03-23T22:33:55Z,0.341125,0.0,0.000000,0.000000,1.124193,0.341125,0.300288
3,user-0001,collab-002-02,project-002,Collaboration 2-2,voting,"[idm, uk-bass]",2026-01-16T13:18:16Z,0.327977,0.0,0.000000,0.000000,0.637364,0.327977,0.170249
4,user-0001,collab-002-03,project-002,Collaboration 2-3,submission,[minimal],2026-03-20T00:06:58Z,0.322133,0.0,0.000000,0.000000,1.516615,0.322133,0.405110


## Sweep


In [6]:
heuristic_mix_values = np.round(np.arange(0.35, 0.86, 0.05), 2)
tag_values = [0.2, 0.3, 0.4, 0.5, 0.6]
project_values = [0.1, 0.2, 0.3, 0.4]
cf_signal_values = [0.7, 0.8, 0.85, 0.9, 1.0]

heuristic_weight_sets = []
for tag_w, project_w in product(tag_values, project_values):
    pop_w = round(1.0 - tag_w - project_w, 2)
    if 0.05 <= pop_w <= 0.5:
        heuristic_weight_sets.append((tag_w, project_w, pop_w))

len(heuristic_mix_values), len(heuristic_weight_sets), len(cf_signal_values), len(heuristic_mix_values) * len(heuristic_weight_sets) * len(cf_signal_values)

(11, 16, 5, 880)

In [7]:
def evaluate_config(heuristic_mix, tag_w, project_w, pop_w, cf_signal_w):
    cf_pop_w = 1.0 - cf_signal_w
    df = candidates.copy()
    df['heuristicScore'] = (
        tag_w * df['tagScoreNorm']
        + project_w * df['projectAffinityNorm']
        + pop_w * df['popularityScoreNorm']
    )
    df['cfScore'] = cf_signal_w * df['cfScoreNorm'] + cf_pop_w * df['cfPopularityNorm']
    df['hybridScore'] = heuristic_mix * df['heuristicScore'] + (1.0 - heuristic_mix) * df['cfScore']

    topk = (
        df.sort_values(['userId', 'hybridScore'], ascending=[True, False])
        .groupby('userId')
        .head(10)
        .copy()
    )
    topk['rank'] = topk.groupby('userId').cumcount() + 1

    hits = topk.merge(holdout_eval, on='userId', how='inner')
    hits['isHit'] = hits['collaborationId'] == hits['heldOutCollaborationId']
    user_hits = hits.groupby('userId').agg(
        hitAt10=('isHit', 'max'),
        reciprocalRank=('rank', lambda s: 1 / s[hits.loc[s.index, 'isHit']].iloc[0] if hits.loc[s.index, 'isHit'].any() else 0),
    ).reset_index()

    topk['tagOverlap'] = [len(set(tags) & history_tags.get(uid, set())) for uid, tags in zip(topk['userId'], topk['tagsKey'])]
    topk['sameSeenProject'] = [pid in history_projects.get(uid, set()) for uid, pid in zip(topk['userId'], topk['projectId'])]
    top1_counts = topk[topk['rank'] == 1]['collaborationId'].value_counts()

    return {
        'heuristic_mix': heuristic_mix,
        'cf_mix': round(1.0 - heuristic_mix, 2),
        'tag_w': tag_w,
        'project_w': project_w,
        'pop_w': pop_w,
        'cf_signal_w': cf_signal_w,
        'cf_pop_w': round(cf_pop_w, 2),
        'hit_rate_at_10': float(user_hits['hitAt10'].mean()),
        'mrr_at_10': float(user_hits['reciprocalRank'].mean()),
        'coverage_pct': 100 * topk['collaborationId'].nunique() / len(collabs),
        'top1_dominance_pct': 100 * (top1_counts.iloc[0] / len(user_hits)),
        'top10_tag_overlap_pct': 100 * (topk['tagOverlap'] > 0).mean(),
        'top10_same_project_seen_pct': 100 * topk['sameSeenProject'].mean(),
    }

results = []
for heuristic_mix in heuristic_mix_values:
    for tag_w, project_w, pop_w in heuristic_weight_sets:
        for cf_signal_w in cf_signal_values:
            results.append(evaluate_config(heuristic_mix, tag_w, project_w, pop_w, cf_signal_w))

results_df = pd.DataFrame(results)
results_df.shape

(880, 13)

## Best by hit rate


In [8]:
results_df.sort_values(['hit_rate_at_10', 'mrr_at_10', 'top1_dominance_pct'], ascending=[False, False, True]).head(20)

,heuristic_mix,cf_mix,tag_w,project_w,pop_w,cf_signal_w,cf_pop_w,hit_rate_at_10,mrr_at_10,coverage_pct,top1_dominance_pct,top10_tag_overlap_pct,top10_same_project_seen_pct
545,0.65,0.35,0.6,0.1,0.3,0.70,0.30,0.426778,0.126205,80.851064,20.083682,94.937238,34.393305
688,0.75,0.25,0.5,0.1,0.4,0.90,0.10,0.422594,0.119956,80.851064,20.083682,94.686192,35.188285
689,0.75,0.25,0.5,0.1,0.4,1.00,0.00,0.414226,0.124577,80.851064,20.083682,94.728033,35.355649
845,0.85,0.15,0.5,0.1,0.4,0.70,0.30,0.414226,0.124246,80.851064,19.246862,95.313808,35.564854
625,0.70,0.30,0.6,0.1,0.3,0.70,0.30,0.414226,0.122712,80.851064,20.502092,95.439331,34.895397
769,0.80,0.20,0.5,0.1,0.4,1.00,0.00,0.414226,0.122508,80.851064,20.083682,95.020921,35.690377
605,0.70,0.30,0.5,0.1,0.4,0.70,0.30,0.414226,0.117587,80.851064,19.246862,93.974895,34.560669
846,0.85,0.15,0.5,0.1,0.4,0.80,0.20,0.410042,0.124744,80.851064,19.665272,95.355649,35.690377
768,0.80,0.20,0.5,0.1,0.4,0.90,0.10,0.410042,0.122322,80.851064,20.502092,95.020921,35.690377
687,0.75,0.25,0.5,0.1,0.4,0.85,0.15,0.410042,0.118991,80.851064,20.083682,94.602510,35.146444


## Best by balance


In [9]:
results_df['balance_score'] = (
    results_df['hit_rate_at_10']
    + 0.5 * results_df['mrr_at_10']
    - 0.0025 * results_df['top1_dominance_pct']
)

results_df.sort_values(['balance_score', 'hit_rate_at_10', 'mrr_at_10'], ascending=[False, False, False]).head(20)

,heuristic_mix,cf_mix,tag_w,project_w,pop_w,cf_signal_w,cf_pop_w,hit_rate_at_10,mrr_at_10,coverage_pct,top1_dominance_pct,top10_tag_overlap_pct,top10_same_project_seen_pct,balance_score
545,0.65,0.35,0.6,0.1,0.3,0.70,0.30,0.426778,0.126205,80.851064,20.083682,94.937238,34.393305,0.439672
688,0.75,0.25,0.5,0.1,0.4,0.90,0.10,0.422594,0.119956,80.851064,20.083682,94.686192,35.188285,0.432363
845,0.85,0.15,0.5,0.1,0.4,0.70,0.30,0.414226,0.124246,80.851064,19.246862,95.313808,35.564854,0.428232
689,0.75,0.25,0.5,0.1,0.4,1.00,0.00,0.414226,0.124577,80.851064,20.083682,94.728033,35.355649,0.426305
769,0.80,0.20,0.5,0.1,0.4,1.00,0.00,0.414226,0.122508,80.851064,20.083682,95.020921,35.690377,0.425271
605,0.70,0.30,0.5,0.1,0.4,0.70,0.30,0.414226,0.117587,80.851064,19.246862,93.974895,34.560669,0.424902
625,0.70,0.30,0.6,0.1,0.3,0.70,0.30,0.414226,0.122712,80.851064,20.502092,95.439331,34.895397,0.424327
846,0.85,0.15,0.5,0.1,0.4,0.80,0.20,0.410042,0.124744,80.851064,19.665272,95.355649,35.690377,0.423251
825,0.85,0.15,0.4,0.1,0.5,0.70,0.30,0.410042,0.116175,78.723404,17.991632,93.179916,35.899582,0.423150
745,0.80,0.20,0.4,0.1,0.5,0.70,0.30,0.405858,0.113027,78.723404,16.736402,92.970711,35.230126,0.420530
